# Notebook 1 - Credit Card Fraud

## 0. Importing modules and downloading Dataset

Setup all the imports.

In [1]:
import kagglehub
import time

import pandas as pd

from sklearn.svm import LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

Instantiate the variables with the dataset name and the paths.

In [3]:
dataset = "mlg-ulb/creditcardfraud"
path = f"../data/{dataset}"
file_path = path + "/creditcard.csv"

Download and load the dataset.

In [4]:
print(f"Downloading dataset '{dataset}' and saving it on '{path}'...")

kagglehub.dataset_download(dataset, output_dir=path)

print("Dataset downloaded!")

Dataset downloaded!


In [5]:
print(f"Loading dataset '{dataset}' and saving it on '{file_path}'...")

df = pd.read_csv(file_path)

print("Dataset loaded!")

Loading dataset 'mlg-ulb/creditcardfraud' and saving it on '../data/mlg-ulb/creditcardfraud/creditcard.csv'...
Dataset loaded!


In [6]:
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


## 1. Preparing data

In [7]:
labels = df["Class"]
data = df.drop("Class", axis=1)

In [8]:
data

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,1.475829,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.059616,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.001396,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.127434,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00


In [9]:
data = StandardScaler().fit_transform(data)
data_train, data_test, labels_train, labels_test = train_test_split(data, labels, test_size=0.2)

In [9]:
labels_train

98515     0
125351    0
88598     0
136348    0
275142    0
         ..
140285    0
247752    0
79170     0
206191    0
229688    0
Name: Class, Length: 227845, dtype: int64

## 2. Training models and collecting data

In [10]:
results_df = pd.DataFrame(columns=["Model Name", "Accuracy", "Precision", "Recall", "F1 Score", "Training Time (s)"])

In [11]:
def print_and_add(model_name, labels_test, results, training_time, results_df):
    tn, fp, fn, tp = confusion_matrix(labels_test, results).ravel()

    accuracy = accuracy_score(labels_test, results)
    precision = precision_score(labels_test, results)
    recall = recall_score(labels_test, results)
    f1 = f1_score(labels_test, results)

    print(f"  Accuracy:       {accuracy:.4f}")
    print(f"  Precision:      {precision:.4f}")
    print(f"  Recall:         {recall:.4f}")
    print(f"  F1 Score:       {f1:.4f}")

    print(f"  True Positives  (TP): {tp:>6}  → Fraud correctly detected")
    print(f"  True Negatives  (TN): {tn:>6}  → Legit correctly identified")
    print(f"  False Positives (FP): {fp:>6}  → Legit flagged as fraud")
    print(f"  False Negatives (FN): {fn:>6}  → Fraud missed by model")

    row = {
        "Model Name":        model_name,
        "Accuracy":          round(accuracy, 4),
        "Precision":         round(precision, 4),
        "Recall":            round(recall, 4),
        "F1 Score":          round(f1, 4),
        "Training Time (s)": round(training_time, 4)
    }

    return pd.concat([results_df, pd.DataFrame([row])], ignore_index=True)

In [12]:
SGD = SGDClassifier()
SGD.fit(data_train, labels_train)

start = time.time()
results = SGD.predict(data_test)    
training_time = time.time() - start

results_df = print_and_add("SGD", labels_test, results, training_time, results_df)

  Accuracy:       0.9991
  Precision:      0.8507
  Recall:         0.5876
  F1 Score:       0.6951
  True Positives  (TP):     57  → Fraud correctly detected
  True Negatives  (TN):  56855  → Legit correctly identified
  False Positives (FP):     10  → Legit flagged as fraud
  False Negatives (FN):     40  → Fraud missed by model


In [13]:
SVC = LinearSVC()
SVC.fit(data_train, labels_train)

start = time.time()
results = SVC.predict(data_test)
training_time = time.time() - start

results_df = print_and_add("Linear SVC", labels_test, results, training_time, results_df)

  Accuracy:       0.9991
  Precision:      0.8485
  Recall:         0.5773
  F1 Score:       0.6871
  True Positives  (TP):     56  → Fraud correctly detected
  True Negatives  (TN):  56855  → Legit correctly identified
  False Positives (FP):     10  → Legit flagged as fraud
  False Negatives (FN):     41  → Fraud missed by model


In [14]:
KNN = KNeighborsClassifier()
KNN.fit(data_train, labels_train)

start = time.time()
results = KNN.predict(data_test)
training_time = time.time() - start

results_df = print_and_add("K-Neighbors Classifier", labels_test, results, training_time, results_df)

  Accuracy:       0.9995
  Precision:      0.9383
  Recall:         0.7835
  F1 Score:       0.8539
  True Positives  (TP):     76  → Fraud correctly detected
  True Negatives  (TN):  56860  → Legit correctly identified
  False Positives (FP):      5  → Legit flagged as fraud
  False Negatives (FN):     21  → Fraud missed by model


In [15]:
DTC = DecisionTreeClassifier()
DTC.fit(data_train, labels_train)

start = time.time()
results = DTC.predict(data_test)
training_time = time.time() - start

results_df = print_and_add("Decision Tree Classifier", labels_test, results, training_time, results_df)

  Accuracy:       0.9993
  Precision:      0.8065
  Recall:         0.7732
  F1 Score:       0.7895
  True Positives  (TP):     75  → Fraud correctly detected
  True Negatives  (TN):  56847  → Legit correctly identified
  False Positives (FP):     18  → Legit flagged as fraud
  False Negatives (FN):     22  → Fraud missed by model


In [16]:
RFC = RandomForestClassifier()
RFC.fit(data_train, labels_train)

results = RFC.predict(data_test)
training_time = time.time() - start

results_df = print_and_add("Random Forest Classifier", labels_test, results, training_time, results_df)

  Accuracy:       0.9996
  Precision:      0.9744
  Recall:         0.7835
  F1 Score:       0.8686
  True Positives  (TP):     76  → Fraud correctly detected
  True Negatives  (TN):  56863  → Legit correctly identified
  False Positives (FP):      2  → Legit flagged as fraud
  False Negatives (FN):     21  → Fraud missed by model


In [17]:
results_df

,Model Name,Accuracy,Precision,Recall,F1 Score,Training Time (s)
0,SGD,0.9991,0.8507,0.5876,0.6951,0.002
1,Linear SVC,0.9991,0.8485,0.5773,0.6871,0.0019
2,K-Neighbors Classifier,0.9995,0.9383,0.7835,0.8539,10.4062
3,Decision Tree Classifier,0.9993,0.8065,0.7732,0.7895,0.0045
4,Random Forest Classifier,0.9996,0.9744,0.7835,0.8686,141.8606


## 3. Training a neural network

In [26]:
def train(net, train_values, train_labels, test_values, test_labels, num_epochs):
    x_train = torch.FloatTensor(train_values)
    x_test  = torch.FloatTensor(test_values)
    y_train = torch.LongTensor(train_labels.values)
    y_test  = torch.LongTensor(test_labels.values)

    # Wrap in DataLoader for batching
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
    test_loader  = DataLoader(TensorDataset(x_test,  y_test),  batch_size=32)

    # Setup
    device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-4)

    # Training loop
    for epoch in range(num_epochs):
        net.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            loss = criterion(net(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        net.eval()
        correct = total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds = net(X_batch).argmax(dim=1)
                correct += (preds == y_batch).sum().item()
                total   += y_batch.size(0)

        print(f"Epoch {epoch+1:>3} | Test acc: {correct/total:.4f}")

In [27]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        self.fc = nn.Sequential(
            nn.Linear(30, 60),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(60, 60),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(60, 2)
        )

    def forward(self, x):
        x = self.fc(x)

        return x

In [28]:
net = Net()
net = train(net, data_train, labels_train, data_test, labels_test, 2)

Epoch   1 | Test acc: 0.9982
Epoch   2 | Test acc: 0.9993
